# DriftWatch - Local XGBoost Training Preview

This notebook previews the training step locally before creating a SageMaker training job.

Workflow:

1. Load the SageMaker-formatted train, validation, and test CSV files.
2. Validate the label and feature schema.
3. Train XGBoost with the planned SageMaker hyperparameters.
4. Inspect validation metrics and early stopping.
5. Evaluate the final model on the held-out test set.
6. Save a local model artifact for later SageMaker integration.

The CSV files are headerless and contain the binary target in the first column.

In [7]:
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "train":
    PROJECT_ROOT = PROJECT_ROOT.parent.parent

DATA_ROOT = PROJECT_ROOT / "driftwatch-bucket" / "data" / "processed"
TRAIN_PATH = DATA_ROOT / "train" / "train.csv"
VALIDATION_PATH = DATA_ROOT / "validate" / "validation.csv"
TEST_PATH = DATA_ROOT / "test" / "test.csv"
MODEL_DIR = PROJECT_ROOT / "artifacts" / "local-training"
MODEL_PATH = MODEL_DIR / "xgboost-model"

RANDOM_STATE = 42
SCALE_POS_WEIGHT = 2.8

print(f"Project root: {PROJECT_ROOT}")
print(f"Training data: {TRAIN_PATH}")
print(f"Validation data: {VALIDATION_PATH}")
print(f"Test data: {TEST_PATH}")
print(f"XGBoost version: {xgb.__version__}")

Project root: /home/sagemaker-user/DriftWatch
Training data: /home/sagemaker-user/DriftWatch/driftwatch-bucket/data/processed/train/train.csv
Validation data: /home/sagemaker-user/DriftWatch/driftwatch-bucket/data/processed/validate/validation.csv
Test data: /home/sagemaker-user/DriftWatch/driftwatch-bucket/data/processed/test/test.csv
XGBoost version: 2.1.4


## 1. Load and validate the processed datasets

The SageMaker XGBoost format has no header and places the target in column zero. This check verifies that all splits have the same feature count and that the target is binary.

In [8]:
def load_sagemaker_csv(path):
    dataframe = pd.read_csv(path, header=None)
    labels = dataframe.iloc[:, 0].astype(int)
    features = dataframe.iloc[:, 1:].astype(float)
    return features, labels


X_train, y_train = load_sagemaker_csv(TRAIN_PATH)
X_validation, y_validation = load_sagemaker_csv(VALIDATION_PATH)
X_test, y_test = load_sagemaker_csv(TEST_PATH)

assert X_train.shape[1] == X_validation.shape[1] == X_test.shape[1]
assert set(y_train.unique()).issubset({0, 1})
assert set(y_validation.unique()).issubset({0, 1})
assert set(y_test.unique()).issubset({0, 1})
assert not X_train.isna().any().any()
assert not X_validation.isna().any().any()
assert not X_test.isna().any().any()

print(f"Train:      {X_train.shape}, churn rate={y_train.mean():.2%}")
print(f"Validation: {X_validation.shape}, churn rate={y_validation.mean():.2%}")
print(f"Test:       {X_test.shape}, churn rate={y_test.mean():.2%}")
print(f"Feature count: {X_train.shape[1]}")

Train:      (4930, 30), churn rate=26.53%
Validation: (1056, 30), churn rate=26.52%
Test:       (1057, 30), churn rate=26.58%
Feature count: 30


## 2. Train the local XGBoost model

These parameters mirror the first fixed SageMaker training job. `aucpr` is used because churn is the minority class, and early stopping uses the validation split to limit overfitting.

In [9]:
train_matrix = xgb.DMatrix(X_train, label=y_train)
validation_matrix = xgb.DMatrix(X_validation, label=y_validation)

evaluation_results = {}
training_parameters = {
    "objective": "binary:logistic",
    "eval_metric": "aucpr",
    "max_depth": 5,
    "eta": 0.1,
    "gamma": 0.0,
    "min_child_weight": 1.0,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "scale_pos_weight": SCALE_POS_WEIGHT,
    "tree_method": "hist",
    "seed": RANDOM_STATE,
}

model = xgb.train(
    params=training_parameters,
    dtrain=train_matrix,
    num_boost_round=300,
    evals=[
        (train_matrix, "train"),
        (validation_matrix, "validation"),
    ],
    evals_result=evaluation_results,
    early_stopping_rounds=15,
    verbose_eval=1,
)

print(f"Best iteration: {model.best_iteration}")
print(f"Best validation PR-AUC: {model.best_score:.4f}")

[0]	train-aucpr:0.61223	validation-aucpr:0.58182
[1]	train-aucpr:0.62217	validation-aucpr:0.59283
[2]	train-aucpr:0.62509	validation-aucpr:0.59602
[3]	train-aucpr:0.63510	validation-aucpr:0.60198
[4]	train-aucpr:0.65629	validation-aucpr:0.61632
[5]	train-aucpr:0.67482	validation-aucpr:0.63718
[6]	train-aucpr:0.67597	validation-aucpr:0.63792
[7]	train-aucpr:0.68784	validation-aucpr:0.63946
[8]	train-aucpr:0.69183	validation-aucpr:0.63676
[9]	train-aucpr:0.69375	validation-aucpr:0.63505
[10]	train-aucpr:0.69811	validation-aucpr:0.64251
[11]	train-aucpr:0.70038	validation-aucpr:0.64359
[12]	train-aucpr:0.70332	validation-aucpr:0.64464
[13]	train-aucpr:0.70638	validation-aucpr:0.64442
[14]	train-aucpr:0.71616	validation-aucpr:0.64482
[15]	train-aucpr:0.71901	validation-aucpr:0.64509
[16]	train-aucpr:0.71900	validation-aucpr:0.64340
[17]	train-aucpr:0.72086	validation-aucpr:0.64171
[18]	train-aucpr:0.72400	validation-aucpr:0.64284
[19]	train-aucpr:0.72596	validation-aucpr:0.64062
[20]	train

## 3. Select a churn threshold on validation data

The default threshold of 0.5 is not assumed to be optimal. We select the threshold using validation labels only, then apply that fixed threshold to the test set.

In [10]:
validation_probabilities = model.predict(
    validation_matrix,
    iteration_range=(0, model.best_iteration + 1),
)

threshold_candidates = np.arange(0.10, 0.91, 0.01)
threshold_scores = []
for threshold in threshold_candidates:
    validation_predictions = (validation_probabilities >= threshold).astype(int)
    threshold_scores.append(
        {
            "threshold": threshold,
            "f1": f1_score(y_validation, validation_predictions, zero_division=0),
            "precision": precision_score(
                y_validation,
                validation_predictions,
                zero_division=0,
            ),
            "recall": recall_score(
                y_validation,
                validation_predictions,
                zero_division=0,
            ),
        }
    )

threshold_table = pd.DataFrame(threshold_scores)
best_threshold_row = threshold_table.loc[threshold_table["f1"].idxmax()]
selected_threshold = float(best_threshold_row["threshold"])

print(f"Selected threshold: {selected_threshold:.2f}")
print(f"Validation F1: {best_threshold_row['f1']:.4f}")
print(f"Validation precision: {best_threshold_row['precision']:.4f}")
print(f"Validation recall: {best_threshold_row['recall']:.4f}")

Selected threshold: 0.60
Validation F1: 0.6410
Validation precision: 0.5637
Validation recall: 0.7429


## 4. Evaluate on the held-out test set

This is the first unbiased estimate after the threshold was selected. PR-AUC is threshold-independent; F1, precision, recall, and the confusion matrix use the validation-selected threshold.

In [11]:
test_matrix = xgb.DMatrix(X_test, label=y_test)
test_probabilities = model.predict(
    test_matrix,
    iteration_range=(0, model.best_iteration + 1),
)
test_predictions = (test_probabilities >= selected_threshold).astype(int)

test_metrics = {
    "pr_auc": average_precision_score(y_test, test_probabilities),
    "f1": f1_score(y_test, test_predictions, zero_division=0),
    "precision": precision_score(y_test, test_predictions, zero_division=0),
    "recall": recall_score(y_test, test_predictions, zero_division=0),
}

print("Test metrics:")
for metric_name, metric_value in test_metrics.items():
    print(f"  {metric_name}: {metric_value:.4f}")

print("\nConfusion matrix [rows=true, columns=predicted]:")
print(confusion_matrix(y_test, test_predictions))

print("\nClassification report:")
print(classification_report(y_test, test_predictions, digits=4, zero_division=0))

Test metrics:
  pr_auc: 0.6856
  f1: 0.6383
  precision: 0.5909
  recall: 0.6940

Confusion matrix [rows=true, columns=predicted]:
[[641 135]
 [ 86 195]]

Classification report:
              precision    recall  f1-score   support

           0     0.8817    0.8260    0.8530       776
           1     0.5909    0.6940    0.6383       281

    accuracy                         0.7909      1057
   macro avg     0.7363    0.7600    0.7456      1057
weighted avg     0.8044    0.7909    0.7959      1057



## 5. Save the local preview artifact

SageMaker will later write the model to `/opt/ml/model/xgboost-model`. This local path mirrors that contract and also records the selected threshold and feature count for the next pipeline steps.

In [12]:
import json

MODEL_DIR.mkdir(parents=True, exist_ok=True)
model.save_model(MODEL_PATH)

metadata = {
    "model_format": "xgboost-json",
    "feature_count": int(X_train.shape[1]),
    "best_iteration": int(model.best_iteration),
    "selected_threshold": selected_threshold,
    "training_parameters": training_parameters,
    "validation_metrics": {
        "pr_auc": float(model.best_score),
        "f1": float(best_threshold_row["f1"]),
        "precision": float(best_threshold_row["precision"]),
        "recall": float(best_threshold_row["recall"]),
    },
    "test_metrics": {name: float(value) for name, value in test_metrics.items()},
}

metadata_path = MODEL_DIR / "training_metadata.json"
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print(f"Saved model: {MODEL_PATH}")
print(f"Saved metadata: {metadata_path}")

Saved model: /home/sagemaker-user/DriftWatch/artifacts/local-training/xgboost-model
Saved metadata: /home/sagemaker-user/DriftWatch/artifacts/local-training/training_metadata.json


/opt/conda/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [20:53:48] WARNING: /home/conda/feedstock_root/build_artifacts/xgboost-split_1744329020674/work/src/c_api/c_api.cc:1374: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  warnings.warn(smsg, UserWarning)
